# 🛵 F-ComFlow — COD Fraud / Return Risk Model

Hi! This notebook is where I built and trained the model that powers the **Cash-on-Delivery (COD) risk score** in F-ComFlow.

**Why this matters:** in Bangladesh most orders are COD, and 20–40% of parcels get refused at the door. The merchant still pays the round-trip courier fee, so every failed delivery hurts. If we can flag a risky order *before* dispatch, the merchant can ask for a small advance payment first.

So the goal is simple: **given what we know about an order, predict how likely it is to fail (be returned).**

I kept this notebook runnable top-to-bottom in Google Colab — just Runtime → Run all.

## What the model looks at

I don't use anything private or fancy — just 5 signals we already have when an order is placed:

| Feature | Meaning |
|---|---|
| `phone_valid` | 1 if the phone is a valid 11-digit BD mobile, else 0 |
| `address_score` | 0–1, how complete the address looks (house/road/area present) |
| `return_rate` | 0–1, this customer's past return rate |
| `past_orders` | how many orders the customer completed before |
| `district_risk` | 0–1, rough regional return risk (Dhaka low, remote areas higher) |

**Label:** `1` = delivery FAILED / returned, `0` = delivered fine.

> Note: these 5 features are exactly what the app sends to the model at runtime (see `ai/app/risk.py`). Keeping them identical is what lets me drop the trained file straight into the app.

In [ ]:
# One-time installs (Colab already has most of these)
!pip -q install xgboost scikit-learn pandas numpy matplotlib joblib

## Step 1 — Build the training data

Real merchant delivery data is hard to get (and private), so I generate a **synthetic 10,000-order dataset**. This is the same logic as `ai/train/generate_dataset.py`.

The trick: I don't label rows randomly. I write down how failures *actually* happen — bad phone, vague address, repeat-returner, risky district all push the failure chance up — then squash that through a logistic curve so risky combos fail *most* of the time and clean orders rarely do. That gives the model a realistic (not perfect) signal to learn.

In [ ]:
import math, random
import pandas as pd
import numpy as np

random.seed(42)   # so the dataset is reproducible
N = 10_000

def make_row():
    phone_valid   = 1 if random.random() < 0.85 else 0
    address_score = round(random.random(), 2)
    past_orders   = random.choice([0, 0, 0, 1, 2, 3, 5, 8, 12])
    return_rate   = round(random.betavariate(1.2, 4), 2) if past_orders > 0 else 0.0
    district_risk = random.choice([0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40])

    # the 'real world' the model has to discover: a weighted risk signal
    raw_risk = (1.0 * (1 - phone_valid)
                + 0.8 * (1 - address_score)
                + 1.2 * return_rate
                + 0.6 * district_risk
                + (0.4 if past_orders == 0 else 0))
    p_fail = 1 / (1 + math.exp(-(4 * raw_risk - 4.5)))   # logistic squash
    label  = 1 if random.random() < p_fail else 0
    return [phone_valid, address_score, return_rate, past_orders, district_risk, label]

cols = ['phone_valid', 'address_score', 'return_rate', 'past_orders', 'district_risk', 'label']
df = pd.DataFrame([make_row() for _ in range(N)], columns=cols)
print(df.shape)
df.head()

In [ ]:
# quick sanity check: how many orders fail, and do the averages make sense?
print('Failure rate:', round(df.label.mean(), 3))
df.groupby('label')[['phone_valid','address_score','return_rate','district_risk']].mean()

Good — failed orders (`label=1`) on average have **lower `phone_valid`**, **lower `address_score`**, and **higher `return_rate`** than delivered ones. That's exactly the pattern we'd expect, so the data isn't nonsense.

## Step 2 — Train / test split

Standard 80/20 split. I only ever look at the test set at the very end, so the AUC I report is honest (the model never saw those rows during training).

In [ ]:
from sklearn.model_selection import train_test_split

FEATURES = ['phone_valid', 'address_score', 'return_rate', 'past_orders', 'district_risk']
X = df[FEATURES]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('train:', X_train.shape[0], ' test:', X_test.shape[0])

## Step 3 — Train the model (XGBoost)

I went with **XGBoost** (gradient-boosted trees). It's a great fit here: small tabular data, non-linear interactions (a bad address matters more for a brand-new customer), and it trains in seconds. If XGBoost isn't available I fall back to scikit-learn's GradientBoosting — same family.

In [ ]:
try:
    from xgboost import XGBClassifier
    model = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
                          eval_metric='logloss', random_state=42)
    kind = 'XGBoost'
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    model = GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42)
    kind = 'GradientBoosting (sklearn fallback)'

model.fit(X_train, y_train)
print('Trained:', kind)

## Step 4 — Does it actually work?

The headline metric is **AUC** (area under the ROC curve) — basically, if I pick a random failed order and a random good order, how often does the model score the failed one higher? 0.5 = coin flip, 1.0 = perfect. Our app's quality bar is **AUC ≥ 0.78**.

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, RocCurveDisplay

proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba)
print(f'Held-out AUC: {auc:.3f}   (bar: >= 0.78)')
print()
print(classification_report(y_test, (proba >= 0.5).astype(int), target_names=['delivered','failed']))

In [ ]:
import matplotlib.pyplot as plt

# Confusion matrix at a 0.5 threshold
cm = confusion_matrix(y_test, (proba >= 0.5).astype(int))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].imshow(cm, cmap='Blues')
ax[0].set_title('Confusion matrix')
ax[0].set_xticks([0,1]); ax[0].set_xticklabels(['delivered','failed'])
ax[0].set_yticks([0,1]); ax[0].set_yticklabels(['delivered','failed'])
ax[0].set_xlabel('predicted'); ax[0].set_ylabel('actual')
for i in range(2):
    for j in range(2):
        ax[0].text(j, i, cm[i, j], ha='center', va='center', fontsize=13)

# ROC curve
RocCurveDisplay.from_predictions(y_test, proba, ax=ax[1])
ax[1].set_title(f'ROC curve (AUC = {auc:.3f})')
plt.tight_layout(); plt.show()

### Which signals matter most?

Nice thing about trees — they tell us what they leaned on. This lines up with intuition: return history and a valid phone do most of the work.

In [ ]:
importances = getattr(model, 'feature_importances_', None)
if importances is not None:
    order = np.argsort(importances)
    plt.figure(figsize=(7, 3.2))
    plt.barh([FEATURES[i] for i in order], importances[order], color='#4f46e5')
    plt.title('Feature importance'); plt.tight_layout(); plt.show()

## Step 5 — Save the model for the app

The app (`ai/app/risk.py`) loads a joblib file shaped like `{model, features, version, auc}`. I save exactly that, so this file drops straight into `ai/models/risk_model_v1.joblib` and the API starts using the ML score instead of the rule fallback.

In [ ]:
import joblib

artifact = {'model': model, 'features': FEATURES, 'version': 'v1', 'auc': round(float(auc), 3)}
joblib.dump(artifact, 'risk_model_v1.joblib')
print('Saved risk_model_v1.joblib')

# In Colab, download it and drop it into the repo at  ai/models/risk_model_v1.joblib
try:
    from google.colab import files
    files.download('risk_model_v1.joblib')
except Exception:
    print('(not on Colab — the file is in the working dir)')

## Step 6 — How it's used in F-ComFlow

Nothing clever at serving time. When an order is about to be confirmed, the Node API gathers the 5 facts and asks the Python service to score them. `risk.py` runs `model.predict_proba(...)`, turns it into a **0–100 score** and a band:

- **LOW** (< 35) — ship it
- **MEDIUM** (35–59) — keep an eye out
- **HIGH** (≥ 60) — ask for an advance payment before booking the courier

Quick check that a clearly-bad order scores high and a clean one scores low:

In [ ]:
def score(phone_valid, address_score, return_rate, past_orders, district_risk):
    row = [[phone_valid, address_score, return_rate, past_orders, district_risk]]
    p = model.predict_proba(row)[0][1]
    pct = round(p * 100)
    band = 'HIGH' if pct >= 60 else 'MEDIUM' if pct >= 35 else 'LOW'
    return pct, band

print('Risky  (bad phone, vague address, repeat returner):', score(0, 0.1, 0.6, 5, 0.35))
print('Clean  (valid phone, full address, loyal customer):', score(1, 0.95, 0.0, 12, 0.10))

## Notes & honest limitations

- **The data is synthetic.** These AUC numbers prove the *pipeline* works; they are a baseline, not a claim about real accuracy. The right next step is to retrain on the merchant's own historical deliveries once there's enough of them — the code doesn't change, only the CSV does.
- **Keep the 5 features in sync.** If I add a feature here, I have to add it in `ai/app/risk.py` too, or the app will send the wrong shape.
- **It's a suggestion, not a blocker.** A HIGH score just nudges the merchant to collect an advance — the order can still go through.
- **Retraining:** rerun this notebook (or `python ai/train/train_model.py`) whenever the data changes; it refuses to save a model below AUC 0.78 so a bad run can't ship.

That's it — the whole COD risk model, start to finish. 🙂